# 🚀 TRACK C: LAST MINUTE GACOR ENSEMBLE
Notebook ini adalah ujung tombak perjuangan kita! Kita akan menggunakan 2 strategi pamungkas sekaligus:
1. **Opsi 1 (Soft Voting):** Gabungan probabilitas murni 6 Model Deep Learning (5 LoRA + 1 Last Layer).
2. **Opsi 2 (Hard Voting 3-Partai):** Sistem pemilu demokratis antara (LoRA 5-Fold) vs (Last Layer) vs (kNN). Jika ketiganya beda, LoRA menang mutlak.

Di akhir, akan tercipta dua file `submission.csv` terkuat yang siap dikirim ke Kaggle!

In [ ]:
import os, sys, shutil, glob
from concurrent.futures import ThreadPoolExecutor
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# 2. Pull Repo Terbaru
REPO_DIR = '/content/satria-data-bdcugm02'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/agaggigit/satria-data-bdcugm02.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# 3. Instalasi Dependency
!pip install -q -U "torchao>=0.16.0"
!pip install -q --no-warn-conflicts -r {REPO_DIR}/track_b/requirements.txt

In [ ]:
# 4. Copy Cepat Gambar TEST ke Storage Lokal Colab (/tmp)
import os
if os.path.exists('/content/drive/MyDrive/BDC2026/test'):
    DRIVE_TEST_DIR = '/content/drive/MyDrive/BDC2026/test'
else:
    DRIVE_TEST_DIR = '/content/drive/MyDrive/BDC2026apace/test'
LOCAL_TEST_DIR = '/tmp/dataset/test'

def copy_img_worker(args):
    src, dst = args
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst) or os.path.getsize(src) != os.path.getsize(dst):
        shutil.copy2(src, dst)

if os.path.exists(DRIVE_TEST_DIR):
    print(f"🚀 Memulai copy cepat gambar TEST dari {DRIVE_TEST_DIR} ke storage lokal...")
    test_imgs = [f for f in glob.glob(os.path.join(DRIVE_TEST_DIR, "*")) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    files_to_copy = [(src, os.path.join(LOCAL_TEST_DIR, os.path.basename(src))) for src in test_imgs]
    
    with ThreadPoolExecutor(max_workers=32) as executor:
        list(executor.map(copy_img_worker, files_to_copy))
    print(f"✅ Selesai meng-copy {len(files_to_copy)} gambar Test ke {LOCAL_TEST_DIR}")
else:
    print(f"⚠️ Folder {DRIVE_TEST_DIR} tidak ditemukan!")


In [ ]:
# 5. Setup Path & Load kNN Submission (Baseline)
sys.path.insert(0, os.path.join(REPO_DIR, 'track_a', 'src'))
sys.path.insert(0, os.path.join(REPO_DIR, 'track_b', 'src'))
sys.path.insert(0, os.path.join(REPO_DIR, 'track_b', 'experiments'))

import torch
import pandas as pd
import numpy as np
import config, lora_ft
from config import CFG

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

KNN_SUBMISSION_PATH = os.path.join(CFG.save_dir, "..", "output_trackC", "submission_apace.csv")
if not os.path.exists(KNN_SUBMISSION_PATH):
    print(f"⚠️ {KNN_SUBMISSION_PATH} tidak ditemukan!")
else:
    knn_df = pd.read_csv(KNN_SUBMISSION_PATH)
    print(f"Loaded kNN submission: {len(knn_df)} baris")


In [ ]:
# 6. Siapkan Test Loader dari Lokal (/tmp)
from transformers import AutoImageProcessor, AutoModel
from dataset import WasteDataset
from transforms import build_transforms
from torch.utils.data import DataLoader

CHECKPOINT = 'google/siglip2-so400m-patch14-384'
processor = AutoImageProcessor.from_pretrained(CHECKPOINT)
data_config = lora_ft.hf_processor_to_data_config(processor)

local_files = glob.glob(os.path.join(LOCAL_TEST_DIR, "*"))
name_to_path = {os.path.splitext(os.path.basename(f))[0]: f for f in local_files}

test_images = []
for img_id in knn_df['id']:
    base_id = str(img_id)
    if base_id.endswith(('.jpg', '.png', '.jpeg')):
        base_id = os.path.splitext(base_id)[0]
    
    if base_id in name_to_path:
        test_images.append(name_to_path[base_id])
    else:
        test_images.append(os.path.join(LOCAL_TEST_DIR, f"{base_id}.jpg"))

test_df = pd.DataFrame({'filepath': test_images, 'label': 0})

eval_tfm = build_transforms(data_config, 384, train=False)
test_ds = WasteDataset(test_df, transform=eval_tfm)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)


In [ ]:
# 7. Fungsi Pembantu Memuat Model
from lora_ft import build_variant
import gc

def load_model_weights(variant, ckpt_path):
    print(f"\n📦 Memuat model {variant} dari {os.path.basename(ckpt_path)}...")
    encoder = AutoModel.from_pretrained(CHECKPOINT).vision_model
    hidden_size = encoder.config.hidden_size
    n_last_blocks = 4 if variant == 'lora' else None
    model, _ = build_variant(variant, encoder, hidden_size, num_classes=3, n_last_blocks=n_last_blocks)
    
    state_dict = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model.load_state_dict(state_dict, strict=False)
    model = model.to(device)
    model.eval()
    return model


In [ ]:
# 8. EKSTRAKSI PROBABILITAS: LoRA 5-Fold
def get_lora_5fold_probs():
    lora_files = [f"lora_ft_fold{i}_5ep_v3_best.pt" for i in range(5)]
    total_samples = len(test_loader.dataset)
    ensemble_probs = torch.zeros((total_samples, 3), device='cpu')
    
    print("\n🔥 Mulai Ekstraksi Probabilitas 5-Fold LoRA...")
    for lf in lora_files:
        ckpt_path = os.path.join(CFG.save_dir, lf)
        if not os.path.exists(ckpt_path):
            print(f"❌ {lf} tidak ditemukan! Pastikan sudah selesai di-training.")
            continue
            
        model = load_model_weights('lora', ckpt_path)
        
        print(f"Mengekstrak fitur dengan {lf}...")
        ptr = 0
        with torch.no_grad():
            for images, _ in test_loader:
                images = images.to(device)
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    logits = model(images)
                probs = torch.softmax(logits.float(), dim=-1).cpu()
                
                batch_len = len(images)
                ensemble_probs[ptr : ptr+batch_len] += probs
                ptr += batch_len
                
        del model
        gc.collect()
        torch.cuda.empty_cache()
        
    return ensemble_probs / 5.0

lora_probs = get_lora_5fold_probs()


In [ ]:
# 9. EKSTRAKSI PROBABILITAS: Last Layer Fold 0
def get_last_layer_probs():
    ckpt_path = os.path.join(CFG.save_dir, "last_layer_ft_fold0_v3_best.pt")
    if not os.path.exists(ckpt_path):
        print(f"\n❌ {ckpt_path} tidak ditemukan!")
        return None
        
    model = load_model_weights('last_layer', ckpt_path)
    total_samples = len(test_loader.dataset)
    ll_probs = torch.zeros((total_samples, 3), device='cpu')
    
    print("Mengekstrak fitur dengan Last Layer Fold 0...")
    ptr = 0
    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(device)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(images)
            probs = torch.softmax(logits.float(), dim=-1).cpu()
            batch_len = len(images)
            ll_probs[ptr : ptr+batch_len] = probs
            ptr += batch_len
            
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return ll_probs

ll_probs = get_last_layer_probs()


In [ ]:
# 10. OPSI 1: SOFT VOTING (Murni Probabilitas Deep Learning)
print("\n==================================================")
print("🌟 MENJALANKAN OPSI 1: SOFT VOTING 6-MODEL")
print("==================================================")

if lora_probs is not None and ll_probs is not None:
    # lora_probs adalah rata-rata dari 5 model. Kita kalikan 5, tambah 1 dari LL, lalu bagi 6.
    total_soft_probs = ((lora_probs * 5.0) + ll_probs) / 6.0
    opsi1_preds = total_soft_probs.argmax(dim=1).numpy()
    
    # Buat submission
    sub1 = knn_df[['id']].copy()
    sub1['predicted'] = opsi1_preds
    
    out_path_1 = os.path.join(CFG.save_dir, "..", "output_trackC", "submission_opsi1_softvoting.csv")
    sub1.to_csv(out_path_1, index=False)
    print(f"✅ Berhasil disimpan: {out_path_1}")
else:
    print("❌ Gagal menjalankan Opsi 1 karena ada probabilitas yang kosong.")


In [ ]:
# 11. OPSI 2: HARD VOTING 3-PARTAI (LoRA vs LL vs kNN)
print("\n==================================================")
print("⚖️ MENJALANKAN OPSI 2: HARD VOTING 3-PARTAI")
print("==================================================")

if lora_probs is not None and ll_probs is not None:
    # Dapatkan prediksi mentah (label)
    pred_lora = lora_probs.argmax(dim=1).numpy()
    pred_ll = ll_probs.argmax(dim=1).numpy()
    pred_knn = knn_df['predicted'].values
    
    opsi2_preds = []
    for l, ll, k in zip(pred_lora, pred_ll, pred_knn):
        # Jika LoRA dan LL sepakat, atau LoRA dan kNN sepakat
        if l == ll or l == k:
            opsi2_preds.append(l)
        # Jika LL dan kNN sepakat menentang LoRA
        elif ll == k:
            opsi2_preds.append(ll)
        # Jika ketiganya berbeda (Tie Breaker dimenangkan oleh kasta tertinggi: LoRA)
        else:
            opsi2_preds.append(l)
            
    # Buat submission
    sub2 = knn_df[['id']].copy()
    sub2['predicted'] = opsi2_preds
    
    out_path_2 = os.path.join(CFG.save_dir, "..", "output_trackC", "submission_opsi2_hardvoting.csv")
    sub2.to_csv(out_path_2, index=False)
    print(f"✅ Berhasil disimpan: {out_path_2}")


In [ ]:
# 12. KESIMPULAN: Beda Opsi 1 dan Opsi 2
if lora_probs is not None and ll_probs is not None:
    beda_opsi = sub1[sub1['predicted'] != sub2['predicted']]
    print("\n==================================================")
    print("🏁 KESIMPULAN AKHIR")
    print("==================================================")
    print(f"Selisih strategi Opsi 1 dan Opsi 2 hanya ada di: {len(beda_opsi)} gambar!")
    
    if len(beda_opsi) > 0:
        print("Daftar perbedaannya (Pilih mana yang mau di-submit ke Kaggle):")
        comparison = beda_opsi.copy()
        comparison = comparison.rename(columns={'predicted': 'Prediksi_Opsi_1'})
        comparison['Prediksi_Opsi_2'] = sub2.loc[beda_opsi.index, 'predicted']
        display(comparison)
    else:
        print("\nLUAR BIASA! Opsi 1 dan Opsi 2 menghasilkan jawaban 100% SAMA PERSIS.")
        print("Kekuatan probabilitas dan logika demokrasi memberikan hasil absolut.")
        print("Pilih file mana saja untuk disubmit, pasti gacor!")
